# Dagua Developer UI Playground

Manual QA notebook for quickly exercising the user-facing API.

Use this notebook to sanity-check:
- graph construction paths
- styling and themes
- layout and rendering variants
- routing and edge labels
- flex constraints and "fix this in place" flows
- JSON and YAML roundtrips
- cinematic exports: poster, optimization animation, tour
- quality-of-life behavior before README examples are updated

In [ ]:
from pathlib import Path
import json

import torch
import dagua
from dagua import (
    DaguaGraph,
    LayoutConfig,
    NodeStyle,
    EdgeStyle,
    ClusterStyle,
    Flex,
    LayoutFlex,
    AlignGroup,
    AnimationConfig,
    TourConfig,
    PosterConfig,
    CameraKeyframe,
)

OUTDIR = Path('/tmp/dagua-ui-playground')
OUTDIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTDIR

## Graph Builders

These intentionally hit the main graph setup surfaces and a few visually stressful motifs.

In [ ]:
def make_product_flow():
    g = DaguaGraph(direction='TB')
    g.add_node('request', label='Incoming Request', type='input')
    g.add_node('clean', label='Normalize + Validate')
    g.add_node('vision', label='Vision Encoder', style=NodeStyle(shape='ellipse'))
    g.add_node('text', label='Text Encoder', style=NodeStyle(shape='ellipse'))
    g.add_node('reason', label='Reasoning Core')
    g.add_node('safety', label='Safety Check', style=NodeStyle(base_color='#E69F00'))
    g.add_node('answer', label='Final Answer', type='output', style=NodeStyle(base_color='#009E73'))
    g.add_edge('request', 'clean', label='intake')
    g.add_edge('clean', 'vision', label='image path', style=EdgeStyle(routing='ortho'))
    g.add_edge('clean', 'text', label='text path', style=EdgeStyle(routing='straight', curvature=0.0))
    g.add_edge('vision', 'reason', label='visual context')
    g.add_edge('text', 'reason', label='language context')
    g.add_edge('reason', 'safety', label='candidate')
    g.add_edge('safety', 'answer', label='approved')
    g.add_cluster('Input Processing', ['clean', 'vision', 'text'], label='Input Processing')
    g.compute_node_sizes()
    return g


def make_edge_list_graph():
    g = DaguaGraph.from_edge_list([
        ('customer', 'classify'),
        ('classify', 'billing'),
        ('classify', 'product'),
        ('classify', 'infra'),
        ('billing', 'resolve'),
        ('product', 'resolve'),
        ('infra', 'resolve'),
        ('resolve', 'kb'),
        ('kb', 'classify'),
    ])
    g.edge_labels = ['route', 'queue', 'queue', 'queue', 'done', 'done', 'done', 'learn', 'feedback']
    g.compute_node_sizes()
    return g


def make_edge_index_graph():
    edge_index = torch.tensor([
        [0, 0, 0, 1, 2, 3, 4, 4, 5],
        [1, 2, 3, 4, 4, 4, 5, 6, 6],
    ], dtype=torch.long)
    g = DaguaGraph.from_edge_index(edge_index, num_nodes=7)
    g.node_labels = ['hub', 'tiny', 'medium', 'long', 'merge', 'tail', 'sink']
    g.node_styles[0] = NodeStyle(shape='ellipse')
    g.node_styles[3] = NodeStyle(shape='diamond')
    g.edge_labels = ['a', 'b', 'c', 'join', 'join', 'join', 'tail', 'late', 'end']
    g.compute_node_sizes()
    return g


def make_nested_clusters_graph():
    g = DaguaGraph.from_edge_list([
        ('input', 'enc.stage0'),
        ('enc.stage0', 'enc.attn'),
        ('enc.attn', 'enc.mlp'),
        ('enc.mlp', 'bridge'),
        ('bridge', 'dec.cross_attn'),
        ('dec.cross_attn', 'dec.ffn'),
        ('dec.ffn', 'output'),
    ])
    idx = {name: i for i, name in enumerate(g.node_labels)}
    g.add_cluster('Encoder', [idx['enc.stage0'], idx['enc.attn'], idx['enc.mlp']], label='Encoder', style=ClusterStyle())
    g.add_cluster('Encoder Inner', [idx['enc.attn'], idx['enc.mlp']], label='Encoder Inner', parent='Encoder', style=ClusterStyle(label_offset=(10.0, 14.0)))
    g.add_cluster('Decoder', [idx['dec.cross_attn'], idx['dec.ffn']], label='Decoder', style=ClusterStyle())
    g.compute_node_sizes()
    return g


def make_flex_graph():
    g = DaguaGraph.from_edge_list([
        ('input', 'a'), ('input', 'b'), ('input', 'c'),
        ('a', 'join'), ('b', 'join'), ('c', 'join'),
        ('join', 'output'),
    ])
    g.compute_node_sizes()
    g.flex = LayoutFlex(
        pins={
            'input': (Flex.locked(0.0), Flex.locked(0.0)),
            'output': (Flex.locked(260.0), Flex.locked(150.0)),
        },
        align_y=[AlignGroup(['a', 'b', 'c'], weight=6.0)],
        node_sep=Flex.firm(60.0),
    )
    return g


GRAPH_BUILDERS = {
    'product_flow': make_product_flow,
    'from_edge_list': make_edge_list_graph,
    'from_edge_index': make_edge_index_graph,
    'nested_clusters': make_nested_clusters_graph,
    'flex_graph': make_flex_graph,
}

GRAPH_BUILDERS.keys()

## Main Render Pad

Change these knobs and rerun to inspect the current default user experience.

In [ ]:
graph_name = 'product_flow'
direction = 'TB'
theme_name = 'default'
steps = 90
edge_opt_steps = 14
device = DEVICE

dagua.reset()
dagua.set_theme(theme_name)

g = GRAPH_BUILDERS[graph_name]()
g.direction = direction
config = LayoutConfig(device=device, direction=direction, steps=steps, edge_opt_steps=edge_opt_steps, seed=42)
fig, ax = dagua.draw(g, config)
fig

## JSON and YAML Roundtrip

Useful for checking the actual persistence surface a user will hit.

In [ ]:
g = make_product_flow()
json_path = OUTDIR / 'product_flow.json'
yaml_path = OUTDIR / 'product_flow.yaml'

dagua.save(g, json_path)
dagua.save(g, yaml_path)
g_json = dagua.load(json_path)
g_yaml = dagua.load(yaml_path)

{
    'json_nodes': g_json.num_nodes,
    'yaml_nodes': g_yaml.num_nodes,
    'json_path': str(json_path),
    'yaml_path': str(yaml_path),
}

## Lower-Level Layout / Route / Render Pad

This is useful when checking whether a problem is in node layout, edge routing, or final drawing.

In [ ]:
g = make_nested_clusters_graph()
config = LayoutConfig(device=device, steps=80, edge_opt_steps=10, direction='TB', seed=42)
positions = dagua.layout(g, config)
g.compute_node_sizes()
curves = dagua.route_edges(positions, g.edge_index, g.node_sizes, g.direction, g)
label_positions = dagua.place_edge_labels(curves, positions, g.node_sizes, g.edge_labels, g)
fig, ax = dagua.render(g, positions, config, curves=curves, label_positions=label_positions)
fig

## "Fix This Visual Feature In Place" Checks

These are the most important advanced manipulations to keep ergonomic.

In [ ]:
g = make_flex_graph()
config = LayoutConfig(device=device, steps=80, edge_opt_steps=-1, seed=42)
fig, ax = dagua.draw(g, config)
fig

In [ ]:
g = make_product_flow()
g.node_styles[g._id_to_index['reason']] = NodeStyle(base_color='#0072B2', min_width=150.0)
g.edge_styles[1] = EdgeStyle(routing='straight', curvature=0.0, color='#D55E00')
g.edge_styles[2] = EdgeStyle(routing='ortho', curvature=0.0, color='#009E73')
config = LayoutConfig(device=device, steps=90, edge_opt_steps=10, seed=42)
fig, ax = dagua.draw(g, config)
fig

## Visual Export Checks

Quickly sanity-check still, optimization, and tour exports without leaving the notebook.

In [ ]:
g = make_product_flow()
config = LayoutConfig(device=device, steps=70, edge_opt_steps=8, seed=42)
positions = dagua.layout(g, config)
poster_path = OUTDIR / 'product_flow_poster.png'
poster_result = dagua.poster(
    g,
    positions=positions,
    config=config,
    output=str(poster_path),
    poster_config=PosterConfig(scene='zoom_pan'),
)
poster_result

In [ ]:
g = make_product_flow()
gif_path = OUTDIR / 'product_flow_optimization.gif'
anim_result = dagua.animate(
    g,
    LayoutConfig(device=device, steps=50, edge_opt_steps=8, seed=42),
    output=str(gif_path),
    animation_config=AnimationConfig(fps=16, dpi=110, max_layout_frames=18, max_edge_frames=8),
)
anim_result

In [ ]:
g = make_nested_clusters_graph()
config = LayoutConfig(device=device, steps=70, edge_opt_steps=8, seed=42)
positions = dagua.layout(g, config)
tour_path = OUTDIR / 'nested_clusters_tour.gif'
tour_result = dagua.tour(
    g,
    positions=positions,
    config=config,
    output=str(tour_path),
    tour_config=TourConfig(
        scene='keyframes',
        keyframes=[
            CameraKeyframe(duration_frames=18, bounds=None, title='Whole graph'),
            CameraKeyframe(duration_frames=20, center_on='enc.attn', scale=0.6, title='Encoder focus'),
            CameraKeyframe(duration_frames=20, center_on='dec.cross_attn', scale=0.6, title='Decoder focus'),
        ],
        fps=18,
    ),
)
tour_result

## Curated Regression Sweep

Rerun this cell before polishing docs or a release to check the overall feel of a few representative graphs.

In [ ]:
examples = [
    ('product_flow', make_product_flow(), 'TB'),
    ('edge_list', make_edge_list_graph(), 'LR'),
    ('nested_clusters', make_nested_clusters_graph(), 'TB'),
]

artifacts = []
for name, graph, direction in examples:
    cfg = LayoutConfig(device=device, direction=direction, steps=70, edge_opt_steps=8, seed=42)
    output = OUTDIR / f'{name}.png'
    dagua.draw(graph, cfg, output=str(output))
    artifacts.append(str(output))

artifacts